In [0]:
%run /Users/scottsbv@gmail.com/student-engagement/utils

In [0]:
# Create leads_processed_contacts from bronze leads_raw
# Calls explode_to_child with dry_run=False to actually WRITE the contacts table + grandchild tables
import time
start = time.time()

print(f"{'='*80}")
print(f"📥 leads_raw → leads_processed_contacts")
print(f"{'='*80}")

# Check CDC watermark for the contacts table
watermark = get_silver_watermark('leads_processed_contacts')
if watermark:
    print(f"  CDC MODE: reading changes after {watermark}")
else:
    print(f"  FULL LOAD: first run")

# Step 1: Parse bronze table
df_flat = parse_bronze_table('leads_raw', watermark=watermark)
if df_flat is None:
    print(f"  ✅ CDC: No new rows since {watermark}")
    # Print existing table details so output matches first run
    all_tables = spark.catalog.listTables("crm_ingestion.silver")
    related = [t for t in all_tables if t.name == 'leads_processed_contacts' or t.name.startswith('leads_processed_contacts_')]
    print(f"\n  📊 Existing silver tables ({len(related)}):")
    for t in related:
        tdf = spark.table(f"crm_ingestion.silver.{t.name}")
        cnt = tdf.count()
        cols = tdf.columns
        print(f"     📦 {t.name} ({len(cols)} cols, {cnt:,} rows): {cols}")
    print(f"\n  📋 LINEAGE TREE:")
    print(f"  # * 1_3_leads_processed_contacts")
    for t in related:
        if t.name != 'leads_processed_contacts':
            print(f"  # * ------1_3_{t.name}")
    print(f"{'='*80}")
else:
    # Step 2: Flatten structs (keep custom struct as-is)
    df_flat = flatten_structs(df_flat, skip_custom=True)

    # Step 3: Find PK and build carry columns
    pk = find_pk(df_flat)
    carry = list(dict.fromkeys(["bronze_insert_date", "bronze_updated_at"] + ([pk] if pk else [])))

    # Step 4: Explode contacts array with dry_run=False to actually write the table
    # This also recursively writes grandchild tables (emails, phones, urls, integration_links)
    if "contacts" in df_flat.columns:
        print(f"\n  🔍 Exploding contacts array → leads_processed_contacts + grandchild tables:")
        explode_to_child(df_flat, 'leads_processed', 'contacts', carry, depth=0, num_path="1_3", parent_pk=pk, dry_run=False)
        print(f"\n  📋 LINEAGE TREE:")
        for num, depth, name in _lineage_tree:
            print(f"  # * {'---' * depth}{num}_{name}")
    else:
        print("  ⚠️ No contacts array found")

    # Clean up temp tables
    for temp in _TEMP_TABLES:
        spark.sql(f"DROP TABLE IF EXISTS {temp}")
    _TEMP_TABLES.clear()

    print(f"\n  ⏱️ Completed in {time.time() - start:.1f}s")
    print(f"{'='*80}")

In [0]:
display(spark.table('crm_ingestion.silver.leads_processed_contacts').limit(10))